In [1]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF Auth OK")

class CFG:
    base_dir = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup')
    stems_dir = base_dir / 'genres_stems'
    noise_dir = base_dir / 'ESC-50-master' / 'audio'
    mashups_dir = base_dir / 'mashups'
    test_csv = base_dir / 'test.csv'
    out_dir = Path('/kaggle/working')
    out_dir.mkdir(exist_ok=True)
    
    sr = 16000
    dur = 10
    n_samples = sr * dur
    full_dur = 30
    full_samples = sr * full_dur
    
    model_name = 'resnet50d'
    epochs = 35
    bs = 96      # Adjusted for higher resolution
    lr = 4e-4    
    
    mixup_p = 0.5
    mixup_alpha = 0.4
    tta_crops = 11  # Denser sliding window
    
    genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
    g2i = {g: i for i, g in enumerate(genres)}
    i2g = {i: g for i, g in enumerate(genres)}

class AudioCache:
    def __init__(self):
        self.stems = {g: [] for g in CFG.genres}
        self.noises = []
        
        print("Caching 16kHz Audio into RAM...")
        for g in tqdm(CFG.genres, desc="Stems"):
            g_path = CFG.stems_dir / g
            for song_dir in g_path.iterdir():
                if song_dir.is_dir():
                    song_stems = {}
                    for st in ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']:
                        p = song_dir / st
                        if p.exists():
                            y, _ = librosa.load(p, sr=CFG.sr, mono=True)
                            song_stems[st] = y.astype(np.float32)
                    if len(song_stems) == 4:
                        self.stems[g].append(song_stems)
                        
        for p in CFG.noise_dir.glob('*.wav'):
            y, _ = librosa.load(p, sr=CFG.sr, duration=5, mono=True)
            if len(y) > 0:
                self.noises.append(y.astype(np.float32))

cache = AudioCache()
gc.collect()

class MashupDataset(Dataset):
    def __init__(self, cache, n_samples, is_train=True):
        self.cache = cache
        self.n = n_samples
        self.is_train = is_train

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        g = random.choice(CFG.genres)
        label = CFG.g2i[g]
        
        songs = random.sample(self.cache.stems[g], 4)
        stems = [songs[0]['bass.wav'], songs[1]['drums.wav'], 
                 songs[2]['other.wav'], songs[3]['vocals.wav']]
        
        mix = np.zeros(CFG.full_samples, dtype=np.float32)
        for s in stems:
            if len(s) < CFG.full_samples:
                s = np.pad(s, (0, CFG.full_samples - len(s)))
            else:
                s = s[:CFG.full_samples]
            
            # Moderately wider volume
            vol = random.uniform(0.5, 1.5) if self.is_train else 1.0
            mix += s * vol

        rms = np.sqrt(np.mean(mix**2)) + 1e-8
        mix = mix * (10**(-20/20) / rms)
        
        if self.is_train:
            noise = random.choice(self.cache.noises)
            snr = random.uniform(8, 20) # Moderate noise
            
            if len(noise) < CFG.full_samples:
                reps = int(np.ceil(CFG.full_samples / len(noise)))
                noise = np.tile(noise, reps)[:CFG.full_samples]
            else:
                st = random.randint(0, len(noise) - CFG.full_samples)
                noise = noise[st:st+CFG.full_samples]
                
            c_pow, n_pow = np.mean(mix**2) + 1e-10, np.mean(noise**2) + 1e-10
            scale = np.sqrt(c_pow / (n_pow * 10**(snr/10)))
            mix += noise * scale

        mix = np.clip(mix, -1.0, 1.0)
        
        st = random.randint(0, CFG.full_samples - CFG.n_samples)
        crop = mix[st:st+CFG.n_samples]
        
        return torch.tensor(crop, dtype=torch.float32), label

def worker_init_fn(worker_id):
    np.random.seed(np.random.get_state()[1][0] + worker_id)
    random.seed(random.randint(0, 2**32 - 1) + worker_id)

class TorchaudioFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # High Resolution: matches AST temporal detail (~1000 frames)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sr, n_fft=512, hop_length=160, f_min=20, f_max=8000, n_mels=128
        )
        self.a2db = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(24)
        self.time_mask = torchaudio.transforms.TimeMasking(48)

    def forward(self, x):
        with torch.cuda.amp.autocast(enabled=False): 
            x = x.float()
            x = self.mel(x)
            x = self.a2db(x)
            
            m = x.mean(dim=[-2, -1], keepdim=True)
            s = x.std(dim=[-2, -1], keepdim=True)
            x = (x - m) / (s + 1e-5)
            
            x = x.unsqueeze(1) # [B, 1, 128, 1001]
            
            if self.training:
                # GPU Fast Tempo Shift
                if random.random() < 0.5:
                    orig_w = x.shape[3]
                    scale = random.uniform(0.85, 1.15)
                    x = F.interpolate(x, scale_factor=(1.0, scale), mode='bilinear', align_corners=False)
                    if x.shape[3] > orig_w:
                        st = random.randint(0, x.shape[3] - orig_w)
                        x = x[:, :, :, st:st+orig_w]
                    elif x.shape[3] < orig_w:
                        x = F.pad(x, (0, orig_w - x.shape[3]))

                # Double SpecAugment
                for _ in range(2):
                    if random.random() < 0.6: x = self.freq_mask(x)
                    if random.random() < 0.6: x = self.time_mask(x)
                
        return x

def train():
    print(f"\n--- Training {CFG.model_name} ---")
    
    train_dl = DataLoader(MashupDataset(cache, 10000, True), batch_size=CFG.bs, shuffle=True, 
                          num_workers=4, pin_memory=True, worker_init_fn=worker_init_fn)
    val_dl = DataLoader(MashupDataset(cache, 1500, False), batch_size=CFG.bs, shuffle=False, 
                        num_workers=4, pin_memory=True, worker_init_fn=worker_init_fn)
    
    fe = TorchaudioFeatureExtractor().to(device)
    model = timm.create_model(CFG.model_name, pretrained=True, in_chans=1, num_classes=10).to(device)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr, total_steps=len(train_dl)*CFG.epochs, pct_start=0.1)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = GradScaler()
    
    best_f1 = 0
    
    for ep in range(CFG.epochs):
        model.train()
        fe.train()
        
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            
            x_feats = fe(x)
            
            if random.random() < CFG.mixup_p:
                lam = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
                idx = torch.randperm(x_feats.size(0)).to(device)
                x_feats = lam * x_feats + (1 - lam) * x_feats[idx]
                y_b = y[idx]
                mix_applied = True
            else:
                mix_applied = False
            
            with autocast():
                logits = model(x_feats)
                if mix_applied:
                    loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y_b)
                else:
                    loss = criterion(logits, y)
                    
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            
        model.eval()
        fe.eval()
        preds, tgts = [], []
        
        with torch.no_grad():
            for x, y in val_dl:
                x = x.to(device)
                x_feats = fe(x)
                with autocast():
                    logits = model(x_feats)
                preds.extend(logits.argmax(1).cpu().numpy())
                tgts.extend(y.numpy())
                
        val_f1 = f1_score(tgts, preds, average='macro')
        print(f"Ep {ep+1}/{CFG.epochs} | Val F1: {val_f1:.4f}")
        
        # NOTE: Val F1 will likely hit 0.99 - 1.000 because we train and validate on the same pool of stems.
        # This is expected and desired to maximize LB score.
        if val_f1 > best_f1:
            best_f1 = val_f1
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state, CFG.out_dir / f'{CFG.model_name}_best.pth')

    print(f"Best Val F1: {best_f1:.4f}")

train()

del cache
gc.collect()

# Inference
print("\n--- Running TTA Inference ---")
test_df = pd.read_csv(CFG.test_csv)
all_preds = np.zeros((len(test_df), 10))

def get_crops(path):
    try:
        y, _ = librosa.load(path, sr=CFG.sr, duration=30, mono=True)
    except:
        y = np.zeros(CFG.full_samples, dtype=np.float32)
        
    if len(y) < CFG.full_samples:
        y = np.pad(y, (0, CFG.full_samples - len(y)))
    else:
        y = y[:CFG.full_samples]
        
    rms = np.sqrt(np.mean(y**2)) + 1e-8
    y = np.clip(y * (10**(-20/20) / rms), -1.0, 1.0)
    
    max_st = CFG.full_samples - CFG.n_samples
    starts = np.linspace(0, max_st, CFG.tta_crops, dtype=int)
    return torch.tensor(np.array([y[s:s+CFG.n_samples] for s in starts]), dtype=torch.float32)

fe = TorchaudioFeatureExtractor().to(device)
fe.eval()

model = timm.create_model(CFG.model_name, pretrained=False, in_chans=1, num_classes=10).to(device)
model.load_state_dict(torch.load(CFG.out_dir / f'{CFG.model_name}_best.pth'))
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.eval()

with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
        crops = get_crops(CFG.mashups_dir / f"song{str(row['id']).zfill(4)}.wav").to(device)
        x_feats = fe(crops)
        with autocast():
            logits = model(x_feats)
        probs = F.softmax(logits, dim=1).mean(dim=0).cpu().numpy()
        all_preds[idx] = probs

test_df['genre'] = [CFG.i2g[i] for i in all_preds.argmax(axis=1)]
test_df['id'] = test_df['id'].astype(str).str.zfill(4)
test_df[['id', 'genre']].to_csv(CFG.out_dir / 'submission.csv', index=False)
print("Inference Complete! Strictly formatted submission.csv saved.")

print("\nDistribution:")
print(test_df['genre'].value_counts().sort_index())

HF Auth OK
Caching 16kHz Audio into RAM...


Stems:   0%|          | 0/10 [00:00<?, ?it/s]


--- Training resnet50d ---


model.safetensors:   0%|          | 0.00/103M [00:00<?, ?B/s]

Ep 1/35 | Val F1: 0.4586
Ep 2/35 | Val F1: 0.8063
Ep 3/35 | Val F1: 0.8703
Ep 4/35 | Val F1: 0.8697
Ep 5/35 | Val F1: 0.9245
Ep 6/35 | Val F1: 0.9467
Ep 7/35 | Val F1: 0.9454
Ep 8/35 | Val F1: 0.9631
Ep 9/35 | Val F1: 0.9690
Ep 10/35 | Val F1: 0.9773
Ep 11/35 | Val F1: 0.9799
Ep 12/35 | Val F1: 0.9794
Ep 13/35 | Val F1: 0.9800
Ep 14/35 | Val F1: 0.9649
Ep 15/35 | Val F1: 0.9845
Ep 16/35 | Val F1: 0.9843
Ep 17/35 | Val F1: 0.9923
Ep 18/35 | Val F1: 0.9859
Ep 19/35 | Val F1: 0.9900
Ep 20/35 | Val F1: 0.9954
Ep 21/35 | Val F1: 0.9973
Ep 22/35 | Val F1: 0.9987
Ep 23/35 | Val F1: 0.9942
Ep 24/35 | Val F1: 0.9966
Ep 25/35 | Val F1: 0.9954
Ep 26/35 | Val F1: 0.9959
Ep 27/35 | Val F1: 0.9986
Ep 28/35 | Val F1: 0.9986
Ep 29/35 | Val F1: 0.9954
Ep 30/35 | Val F1: 0.9975
Ep 31/35 | Val F1: 0.9987
Ep 32/35 | Val F1: 0.9993
Ep 33/35 | Val F1: 0.9993
Ep 34/35 | Val F1: 0.9972
Ep 35/35 | Val F1: 0.9986
Best Val F1: 0.9993

--- Running TTA Inference ---


Inference:   0%|          | 0/3020 [00:00<?, ?it/s]

Inference Complete! Strictly formatted submission.csv saved.

Distribution:
genre
blues        295
classical    311
country      262
disco        308
hiphop       306
jazz         293
metal        314
pop          312
reggae       304
rock         315
Name: count, dtype: int64
